📌 Procurement and Transportation Optimization Problem – Documentation

🔹 Required File: Updated_Procurement_and_Transport_Costs_v2.csv – Main data file containing information on procurement locations, transportation routes, costs, etc.

This file is derived from the original and follows the logic:
Country A procures Product A → Port A → Warehouse A → Disaster Zones A, B, C



Cost Structure:



Procurement Cost


Transportation Cost (Port → Warehouse, Warehouse → Disaster Zone)



Handling Cost (at Ports & Warehouses, see Nodes sheet column: Handling cost ($/ton))



You can directly reference the df_updated dataframe to view the table data.

🔹 Required File: Data OPT for DS.xlsx – Supplementary data file including warehouse/port capacity and disaster zone demand information.

🔹 Objective:
To determine the optimal procurement and transportation plan that minimizes total cost while satisfying the nutritional needs of each disaster zone.

Total Cost = Procurement Cost + Transportation Cost (Sea + Land) + Handling Cost

Optimization Variable:
The quantity (in tons) procured and transported along each feasible route.
For example: Procuring N tons of chickpeas from Australia and shipping them to MOMBASA (KENYA) port, NAZARETH warehouse, and then to the Afar disaster zone.
Repeat this logic for all disaster zones.

🔹 Key Constraints:

Supplier Capacity Constraint: Cannot exceed the maximum procurement capacity of each supplier.

Port Throughput Constraint: Total incoming volume at each port must not exceed its maximum throughput. (See “Nodes” sheet column: Port capacity (mt/month))

Warehouse Throughput Constraint: Total incoming volume at each warehouse must not exceed its maximum throughput. (See “Nodes” sheet column: Port capacity (mt/month))

Disaster Zone Nutritional Demand: All food allocations must meet the monthly nutritional requirements of each disaster zone.

⚠️ Assumption: Since the original problem does not mention storage capacity at ports and warehouses, it is assumed that all procured goods are directly shipped to disaster zones.

⚠️ Data Precision: Some original data contain long decimal values. Instead of rounding them, calculations use raw data to support later integer optimization.

🔹 Expected Output File:

optimized_results.csv including the following columns:

Country (Supplier Country)

Commodity (Product Type)

Port (Destination Port)

Warehouse (Transit Warehouse)

Camp (Final Disaster Zone)

Optimized Quantity (Optimal Procurement Quantity in tons)

Total Cost per ton (Unit Cost)

Total Cost (Final Cost)

The final file only displays decisions with non-zero procurement quantities, i.e., which supplier countries procured which commodities, transported to which disaster zones, and the corresponding total cost. 🚀








In [1]:
!pip install openpyxl
!pip install cvxpy

In [2]:
import pandas as pd
import numpy as np
import cvxpy as cp

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Specify the correct path to your files in Google Drive
data_opt_path = '/content/drive/MyDrive/Colab Notebooks/Data OPT for DS.xlsx'
procurement_costs_path = '/content/drive/MyDrive/Colab Notebooks/Updated_Procurement_and_Transport_Costs_v2.csv'



Mounted at /content/drive


In [4]:
# Read the Excel file

xls = pd.ExcelFile(data_opt_path)
# Load the newly uploaded CSV file

df_updated = pd.read_csv(procurement_costs_path)


In [5]:
df_updated

,Commodity,Country,Procurement Price,Port,SeaTransport Price,Warehouse,Port to Warehouse Price,Camp,Warehouse to Camp Price,Port Handling Cost,Warehouse Handling Cost,Total Cost
0,CHICKPEAS,AUSTRALIA,696.263100,MOMBASA (KENYA),242.0,<NAZARETH>,308.46800,Afar,88.461600,30.0,11.71432,1376.907020
1,CHICKPEAS,AUSTRALIA,696.263100,MOMBASA (KENYA),242.0,<NAZARETH>,308.46800,Amhara,50.713000,30.0,11.71432,1339.158420
2,CHICKPEAS,AUSTRALIA,696.263100,MOMBASA (KENYA),242.0,<NAZARETH>,308.46800,Benshangul,58.369200,30.0,11.71432,1346.814620
3,CHICKPEAS,AUSTRALIA,696.263100,MOMBASA (KENYA),242.0,<NAZARETH>,308.46800,Gambella,46.779850,30.0,11.71432,1335.225270
4,CHICKPEAS,AUSTRALIA,696.263100,MOMBASA (KENYA),242.0,<NAZARETH>,308.46800,Oromiya,63.868500,30.0,11.71432,1352.313920
...,...,...,...,...,...,...,...,...,...,...,...,...
11829,SUGAR,UTD.ARAB EMIR.,505.162439,PORT SUDAN (SUDAN),198.0,<NAZARETH>,116.05321,TERKIEDI,91.102667,23.5,11.71432,945.532636
11830,SUGAR,UTD.ARAB EMIR.,505.162439,PORT SUDAN (SUDAN),198.0,<NAZARETH>,116.05321,OKUGO,75.563161,23.5,11.71432,929.993129
11831,SUGAR,UTD.ARAB EMIR.,505.162439,PORT SUDAN (SUDAN),198.0,<NAZARETH>,116.05321,AYSAITA,17.594363,23.5,11.71432,872.024332
11832,SUGAR,UTD.ARAB EMIR.,505.162439,PORT SUDAN (SUDAN),198.0,<NAZARETH>,116.05321,DILLO,69.656769,23.5,11.71432,924.086738


In [6]:
# Read all sheets from the Excel file

df_nodes = pd.read_excel(xls, sheet_name="Nodes")
df_commodities = pd.read_excel(xls, sheet_name="Commodities")
df_nutrients = pd.read_excel(xls, sheet_name="Nutrients")
df_nutritional_values = pd.read_excel(xls, sheet_name="Nutritional values")
df_procurement = pd.read_excel(xls, sheet_name="Procurement")
df_SeaTransport = pd.read_excel(xls, sheet_name="SeaTransport")
df_LandTransport = pd.read_excel(xls, sheet_name="LandTransport")

/usr/local/lib/python3.11/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.11/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.11/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.11/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.11/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.11/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.11/dist-packages/openpyxl/worksheet/_reader.py:

In [7]:
# Extract unique names of each location type from the 'df_nodes' DataFrame

port_list = df_nodes.loc[df_nodes['LocationTYpe'] == 'Port', 'Location'].unique()
warehouse_list=df_nodes.loc[df_nodes['LocationTYpe'] == 'Warehouse', 'Location'].unique()
Supplier_list=df_nodes.loc[df_nodes['LocationTYpe'] == 'Supplier', 'Location'].unique()
camp_list=df_nodes.loc[df_nodes['LocationTYpe'] == 'Beneficiary Camp', 'Location'].unique()

In [11]:
# Select data where LocationType is "Beneficiary Camp"
df_beneficiary_camps = df_nodes[df_nodes["LocationTYpe"] == "Beneficiary Camp"]

# Extract the number of beneficiaries in each disaster camp
df_beneficiary_camps = df_beneficiary_camps[["Location", "#Beneficiaries"]].dropna()

# Ensure correct data format (population as integers)
df_beneficiary_camps["# Beneficiaries"] = df_beneficiary_camps["#Beneficiaries"].astype(int)

# First ensure the data format is correct
df_nutrients = df_nutrients.set_index("Nutrient")  # Set the index of the nutrition requirement table to "Nutrient"
df_beneficiaries = df_beneficiary_camps.set_index("Location")["#Beneficiaries"]  # Number of beneficiaries in each camp

# Calculate the monthly nutritional requirement for each camp
df_nutrition_demand_monthly = df_beneficiaries.values[:, None] * df_nutrients.T.values * 30

# Convert to DataFrame for easier viewing
df_nutrition_demand_monthly = pd.DataFrame(df_nutrition_demand_monthly,
                                           columns=df_nutrients.index,
                                           index=df_beneficiaries.index)

df_nutrition_demand_monthly

Nutrient,Energy,Protein,Fat,Calcium,Iron,Iodine,Vit. A,Thiamine,Riboflavin,Niacin,Vit. C
Location,,,,,,,,,,,
Somali,5.265144e+10,1.316286e+09,1.002885e+09,1.128245e+10,5.515865e+08,3.760817e+09,1.253606e+10,2.256490e+07,3.510096e+07,3.474995e+08,7.020192e+08
ADI-HARUSH,1.356012e+09,3.390030e+07,2.582880e+07,2.905740e+08,1.420584e+07,9.685800e+07,3.228600e+08,5.811480e+05,9.040080e+05,8.949679e+06,1.808016e+07
AW-BARRE,1.741194e+09,4.352985e+07,3.316560e+07,3.731130e+08,1.824108e+07,1.243710e+08,4.145700e+08,7.462260e+05,1.160796e+06,1.149188e+07,2.321592e+07
AYSAITA,1.284633e+09,3.211582e+07,2.446920e+07,2.752785e+08,1.345806e+07,9.175950e+07,3.058650e+08,5.505570e+05,8.564220e+05,8.478578e+06,1.712844e+07
BAMBASI,1.987902e+09,4.969755e+07,3.786480e+07,4.259790e+08,2.082564e+07,1.419930e+08,4.733100e+08,8.519580e+05,1.325268e+06,1.312015e+07,2.650536e+07
BERHALE,1.113336e+09,2.783340e+07,2.120640e+07,2.385720e+08,1.166352e+07,7.952400e+07,2.650800e+08,4.771440e+05,7.422240e+05,7.348018e+06,1.484448e+07
DILLO,1.580040e+08,3.950100e+06,3.009600e+06,3.385800e+07,1.655280e+06,1.128600e+07,3.762000e+07,6.771600e+04,1.053360e+05,1.042826e+06,2.106720e+06
HITSATS,1.476342e+09,3.690855e+07,2.812080e+07,3.163590e+08,1.546644e+07,1.054530e+08,3.515100e+08,6.327180e+05,9.842280e+05,9.743857e+06,1.968456e+07
KEBRIBAYA,1.994517e+09,4.986292e+07,3.799080e+07,4.273965e+08,2.089494e+07,1.424655e+08,4.748850e+08,8.547930e+05,1.329678e+06,1.316381e+07,2.659356e+07


In [12]:
# Pivot table to reshape data into "Commodity × Nutrient" format
df_nutritional_values_per_ton = df_nutritional_values.pivot(index="Commodity", columns="Nutrient",
                                                             values="Nutritional value (value/100 gram)!!")

# Convert units (per 100g → per ton)
df_nutritional_values_per_ton = df_nutritional_values_per_ton * 10000

df_nutritional_values_per_ton


Nutrient,Calcium,Energy,Fat,Iodine,Iron,Niacin,Protein,Riboflavin,Thiamine,Vit. A,Vit. C
Commodity,,,,,,,,,,,
Beans,1.473846e+06,3.444438e+06,26130.769231,4.615385e+03,75261.538462,57846.923077,241630.769231,2801.538462,6157.692308,2.715385e+04,3.630769e+04
Chickpeas,1.050000e+06,3.640000e+06,60400.000000,0.000000e+00,62400.000000,46243.333333,193000.000000,2120.000000,4770.000000,2.010000e+05,4.000000e+04
Corn Soya Blend,4.396943e+06,3.842529e+06,89671.428571,4.805714e+05,91600.000000,111228.571429,157600.000000,7442.857143,5628.571429,5.483900e+06,1.011771e+06
High Energy Biscuits,2.500000e+06,4.000000e+06,150000.000000,7.500000e+05,110000.000000,60000.000000,90000.000000,0.000000,0.000000,2.500000e+06,2.000000e+05
Iodised Salt,0.000000e+00,0.000000e+00,0.000000,6.000000e+07,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
Lentils,5.100000e+05,3.380000e+06,10000.000000,0.000000e+00,90200.000000,68033.333333,281000.000000,2500.000000,4800.000000,1.171171e+05,6.000000e+04
Maize,7.333333e+04,2.620000e+06,30600.000000,0.000000e+00,19800.000000,17633.333333,77400.000000,1540.000000,3233.333333,7.510000e+05,2.266667e+04
Rice,9.590909e+04,3.623636e+06,5436.363636,0.000000e+00,35181.818182,58270.000000,70000.000000,377.272727,2577.272727,5.454545e+05,0.000000e+00
Sorghum/Millet,2.600000e+05,3.350000e+06,30000.000000,0.000000e+00,45000.000000,49960.000000,110000.000000,1500.000000,3400.000000,0.000000e+00,0.000000e+00


In [13]:

# Define the decision variable for each row in the dataframe
X = {
    index: cp.Variable(nonneg=True) for index in df_updated.index
}

# Display a summary of variable creation
len(X), list(X.keys())[:5]  # Display first few keys for verification


(11834, [0, 1, 2, 3, 4])

In [14]:
constraints=[]

In [15]:
# Constraint 1: Procurement capacity limitation
for _, row in df_procurement.iterrows():
    country = row["Supplier"]
    commodity = row["Commodity"]
    max_procurement = row["Procurement capacity (ton/month)"]

    # Find all variable indices related to this country and commodity
    indices = df_updated[(df_updated["Country"] == country) & (df_updated["Commodity"] == commodity)].index

    # Calculate total procurement amount for this commodity
    total_procurement = sum(X[idx] for idx in indices)

    # Add procurement capacity constraint
    constraints.append(total_procurement <= max_procurement)


In [16]:
# Constraint 2: Port throughput limitation
# Correctly filter data to include only ports, ensuring warehouses are excluded
ports_df = df_nodes[(df_nodes["LocationTYpe"] == "Port") & (df_nodes["Port capacity (mt/month)"].notna())]

# Iterate through all ports and add throughput constraints
for _, row in ports_df.iterrows():
    port = row["Location"]  # Ensure only ports are selected

    max_capacity_port = row["Port capacity (mt/month)"]

    # Select all variables passing through this port
    indices = df_updated[df_updated["Port"] == port].index

    # Filter out indices not present in `X` to avoid KeyError
    valid_indices = [idx for idx in indices if idx in X]

    # Calculate total flow through the port
    total_port_flow = sum(X[idx] for idx in valid_indices)

    # Add port throughput constraint
    constraints.append(total_port_flow <= max_capacity_port)





In [17]:
# Constraint 3: Warehouse throughput limitation

# Correctly filter data to include only warehouses, ensuring other location types are excluded
warehouses_df = df_nodes[(df_nodes["LocationTYpe"] == "Warehouse") & (df_nodes["Handling cost ($/ton)"].notna())]

# Iterate through all warehouses and add throughput constraints
for _, row in warehouses_df.iterrows():
    warehouse = row["Location"]  # Ensure only warehouses are selected

    max_capacity_warehouse = row["Port capacity (mt/month)"]  # Stores the warehouse's throughput capacity

    # Select all variables passing through this warehouse
    indices = df_updated[df_updated["Warehouse"] == warehouse].index

    # Filter out indices not present in `X` to avoid KeyError
    valid_indices = [idx for idx in indices if idx in X]

    # Calculate total flow through the warehouse
    total_warehouse_flow = sum(X[idx] for idx in valid_indices)

    # Add warehouse throughput constraint
    constraints.append(total_warehouse_flow <= max_capacity_warehouse)



In [18]:
# Constraint 4: Each disaster zone must receive enough food to meet its monthly nutritional needs

# Standardize commodity names by converting all to uppercase
df_updated["Commodity"] = df_updated["Commodity"].str.upper()
df_nutritional_values_per_ton.index = df_nutritional_values_per_ton.index.str.upper()

# Check for mismatched commodities
mismatched_commodities = set(df_updated["Commodity"].unique()) - set(df_nutritional_values_per_ton.index)
print("⚠️ The following commodities exist in df_updated but are missing in df_nutritional_values_per_ton:", mismatched_commodities)

# If mismatches still exist, check for extra spaces
df_updated["Commodity"] = df_updated["Commodity"].str.strip()
df_nutritional_values_per_ton.index = df_nutritional_values_per_ton.index.str.strip()

# Re-check for mismatches after stripping spaces
mismatched_commodities_after_strip = set(df_updated["Commodity"].unique()) - set(df_nutritional_values_per_ton.index)
print("⚠️ Commodities still unmatched after stripping spaces:", mismatched_commodities_after_strip)

# Now the constraints code should work correctly
for nutrient in df_nutritional_values_per_ton.columns[1:]:  # Exclude the "Commodity" column
    for camp in df_nutrition_demand_monthly.index:  # Iterate over all disaster camps
        # Calculate total nutrient supply for this camp
        total_nutrient_supply = sum(
            X[idx] * df_nutritional_values_per_ton.loc[df_updated.loc[idx, "Commodity"], nutrient]
            for idx in df_updated.index
            if df_updated.loc[idx, "Camp"] == camp and df_updated.loc[idx, "Commodity"] in df_nutritional_values_per_ton.index
        )

        # Add constraint to ensure supply ≥ demand
        constraints.append(total_nutrient_supply >= df_nutrition_demand_monthly.loc[camp, nutrient])

# Output success message
print("✅ Nutritional demand constraints successfully added!")



⚠️ The following commodities exist in df_updated but are missing in df_nutritional_values_per_ton: {'SORGHUM/MILLET', 'CHICKPEAS', 'MAIZE', 'LENTILS', 'SUGAR', 'VEGETABLE OIL', 'RICE', 'IODISED SALT', 'SPLIT PEAS', 'WHEAT FLOUR', 'WHEAT', 'CORN SOYA BLEND', 'BEANS'}
⚠️ Commodities still unmatched after stripping spaces: set()
✅ Nutritional demand constraints successfully added!


In [19]:
objective = cp.Minimize(sum(X[idx] * df_updated.loc[idx, "Total Cost"] for idx in df_updated.index))


In [20]:
# Define the optimization problem
problem = cp.Problem(objective, constraints)

# Solve the optimization problem
problem.solve(verbose=True)





/usr/local/lib/python3.11/dist-packages/cvxpy/problems/problem.py:158: UserWarning: Objective contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  warnings.warn("Objective contains too many subexpressions. "


                                     CVXPY                                     
                                     v1.6.5                                    
(CVXPY) May 01 11:06:27 AM: Your problem has 11834 variables, 427 constraints, and 0 parameters.
(CVXPY) May 01 11:06:30 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) May 01 11:06:30 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) May 01 11:06:30 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) May 01 11:06:30 AM: Your problem is compiled with the CPP canonicalization backend.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) May 01 11:06:35 AM: Compiling problem (target solver=CLARAB

/usr/local/lib/python3.11/dist-packages/cvxpy/problems/problem.py:1504: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


np.float64(36050838.765927225)

In [21]:
optimized_results = pd.DataFrame({
    "Commodity": [df_updated.loc[idx, "Commodity"] for idx in df_updated.index],
    "Country": [df_updated.loc[idx, "Country"] for idx in df_updated.index],
    "Port": [df_updated.loc[idx, "Port"] for idx in df_updated.index],
    "Warehouse": [df_updated.loc[idx, "Warehouse"] for idx in df_updated.index],
    "Camp": [df_updated.loc[idx, "Camp"] for idx in df_updated.index],
    "Optimized Quantity (tons)": [X[idx].value if X[idx].value is not None else 0 for idx in df_updated.index],
    "Total Cost per ton": [df_updated.loc[idx, "Total Cost"] for idx in df_updated.index],
    "Total Cost": [(X[idx].value if X[idx].value is not None else 0) * df_updated.loc[idx, "Total Cost"] for idx in df_updated.index]
})

# Filter out items that were not procured (keep only rows with quantity > 0)
optimized_results = optimized_results[optimized_results["Optimized Quantity (tons)"] > 0]


In [22]:
optimized_results

,Commodity,Country,Port,Warehouse,Camp,Optimized Quantity (tons),Total Cost per ton,Total Cost
0,CHICKPEAS,AUSTRALIA,MOMBASA (KENYA),<NAZARETH>,Afar,2.123557e-07,1376.907020,0.000292
1,CHICKPEAS,AUSTRALIA,MOMBASA (KENYA),<NAZARETH>,Amhara,2.295744e-07,1339.158420,0.000307
2,CHICKPEAS,AUSTRALIA,MOMBASA (KENYA),<NAZARETH>,Benshangul,2.355437e-07,1346.814620,0.000317
3,CHICKPEAS,AUSTRALIA,MOMBASA (KENYA),<NAZARETH>,Gambella,2.353698e-07,1335.225270,0.000314
4,CHICKPEAS,AUSTRALIA,MOMBASA (KENYA),<NAZARETH>,Oromiya,2.262967e-07,1352.313920,0.000306
...,...,...,...,...,...,...,...,...
11829,SUGAR,UTD.ARAB EMIR.,PORT SUDAN (SUDAN),<NAZARETH>,TERKIEDI,3.947326e-07,945.532636,0.000373
11830,SUGAR,UTD.ARAB EMIR.,PORT SUDAN (SUDAN),<NAZARETH>,OKUGO,3.929309e-07,929.993129,0.000365
11831,SUGAR,UTD.ARAB EMIR.,PORT SUDAN (SUDAN),<NAZARETH>,AYSAITA,3.825647e-07,872.024332,0.000334
11832,SUGAR,UTD.ARAB EMIR.,PORT SUDAN (SUDAN),<NAZARETH>,DILLO,3.922637e-07,924.086738,0.000362
